#### System Dependencies  
To get started with Unstructured.io, we need a few system-wide dependencies:

#### Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

#### Tesseract (tesseract-ocr)
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

#### libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

##### winget install UB-Mannheim.TesseractOCR
##### winget install oschwartz10612.Poppler

In [1]:
%pip install -Uq "unstructured[pdf]" 
%pip install -Uq langchain_chroma 
%pip install -Uq langchain langchain-community
%pip install -Uq groq 
%pip install -Uq python_dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from typing import List
import os 

#Unstructured for document parsing 
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"batch_size": 32}
)

load_dotenv()
import google.generativeai as genai
import PIL.Image
import base64
import io
genai.configure(api_key=os.getenv("GEMINI_API_KEY")) # initialize the Groq client with the API key from the .env file7


c:\Users\Dell\Desktop\RAG-LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Dell\AppData\Local\Temp\ipykernel_10480\3758826166.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
C:\Users\Dell\AppData\Local\Temp\ipykernel_10480\3758826166.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as

In [3]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

In [4]:
# Test with your PDF file
file_path = "../data/attention-is-all-you-need.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: ../data/attention-is-all-you-need.pdf


No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 2926.06it/s]


✅ Extracted 266 elements


In [5]:
elements

In [6]:

# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Formula'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [7]:
elements[30].to_dict()

{'type': 'NarrativeText',
 'element_id': '3071bbda7672f831ca7f0e4c2efb4bd5',
 'text': 'illia.polosukhin@gmail.com',
 'metadata': {'detection_class_prob': 0.5035039782524109,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(1154.3458251953125),
     np.float64(1681.2060280555554)),
    (np.float64(1154.3458251953125), np.float64(1729.635333611111)),
    (np.float64(1823.7408447265625), np.float64(1729.635333611111)),
    (np.float64(1823.7408447265625), np.float64(1681.2060280555554))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-06-14T18:45:27',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': '../data',
  'filename': 'attention-is-all-you-need.pdf',
  'parent_id': 'b3b03d9842413a17b2859f7f2b274a0d'}}

In [8]:
images = [element for element in elements if element.category == "Image"]
print(f"found {len(images)} images")

images[0].to_dict()

found 7 images


{'type': 'Image',
 'element_id': 'cd522e329946ebd9538f9ed81a4e9348',
 'text': 'Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx | -Casda Nom] Add & Norm VWEeea Multi-Head Multi-Head Attention Attention Sy ae, Se a, Positional @ Encoding @ Positional @ q Encoding Input Embedding Inputs Outputs (shifted right) Output Embedding',
 'metadata': {'coordinates': {'points': ((np.float64(955.4951388888888),
     np.float64(350.00972222222197)),
    (np.float64(955.4951388888888), np.float64(1917.309722222222)),
    (np.float64(2019.4951388888885), np.float64(1917.309722222222)),
    (np.float64(2019.4951388888885), np.float64(350.00972222222197))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-06-14T18:45:27',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': '.

In [9]:
tables = [element for element in elements if element.category == "Table"]
print (f"found {len(tables)} tables")

tables[0].to_dict()

found 4 tables


{'type': 'Table',
 'element_id': 'a89281d85d3d589d6b9b2c889815cbce',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9282334446907043,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(561.3707885742188),
     np.float64(550.45751953125)),
    (np.float64(561.3707885742188), np.float64(908.8243408203125)),
    (np.float64(2395.140380859375), np.float64(908.8243408203125)),
    (np.float64(2395.140380859375), np.float64(550.45751953125))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-06-14T18:45:27',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Att

In [10]:

def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

chunks

🔨 Creating smart chunks...
✅ Created 33 chunks


In [11]:
# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>",
 "<class 'unstructured.documents.elements.Table'>"}

In [12]:
# view one single chunk
    #chunks[2].to_dict()

# view the original atomic elements
#chunks[4].metadata.orig_elements

# view one single original atomic element
chunks[4].metadata.orig_elements[3].to_dict()
    

{'type': 'Image',
 'element_id': 'cd522e329946ebd9538f9ed81a4e9348',
 'text': 'Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx | -Casda Nom] Add & Norm VWEeea Multi-Head Multi-Head Attention Attention Sy ae, Se a, Positional @ Encoding @ Positional @ q Encoding Input Embedding Inputs Outputs (shifted right) Output Embedding',
 'metadata': {'coordinates': {'points': ((np.float64(955.4951388888888),
     np.float64(350.00972222222197)),
    (np.float64(955.4951388888888), np.float64(1917.309722222222)),
    (np.float64(2019.4951388888885), np.float64(1917.309722222222)),
    (np.float64(2019.4951388888885), np.float64(350.00972222222197))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-06-14T18:45:27',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': '.

In [13]:
chunks[4].metadata.orig_elements[3].to_dict()


{'type': 'Image',
 'element_id': 'cd522e329946ebd9538f9ed81a4e9348',
 'text': 'Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx | -Casda Nom] Add & Norm VWEeea Multi-Head Multi-Head Attention Attention Sy ae, Se a, Positional @ Encoding @ Positional @ q Encoding Input Embedding Inputs Outputs (shifted right) Output Embedding',
 'metadata': {'coordinates': {'points': ((np.float64(955.4951388888888),
     np.float64(350.00972222222197)),
    (np.float64(955.4951388888888), np.float64(1917.309722222222)),
    (np.float64(2019.4951388888885), np.float64(1917.309722222222)),
    (np.float64(2019.4951388888885), np.float64(350.00972222222197))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-06-14T18:45:27',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': '.

In [14]:
def seperate_content_types(chunk):
    """Analyze what types of content the chunk contains"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            match element_type:
                case 'Image':
                    # Bug fix: was appending 'table' instead of 'image'
                    content_data['types'].append('image')
                    if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                        content_data['images'].append(element.metadata.image_base64)
                case 'Table':
                    if hasattr(element, 'metadata') and hasattr(element.metadata, 'text_as_html'):
                        content_data['types'].append('table')
                        content_data['tables'].append(element.metadata.text_as_html)

    content_data['types'] = list(set(content_data['types']))
    return content_data


def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""

    prompt_text = f"""You are creating a searchable description for document content retrieval.

CONTENT TO ANALYZE:
TEXT CONTENT:
{text}

"""

    if tables:
        prompt_text += "TABLES:\n"
        for i, table in enumerate(tables):
            prompt_text += f"Table {i+1}:\n{table}\n\n"

    # Bug fix: moved outside the loop so it appears once
    prompt_text += """
YOUR TASK:
Generate a comprehensive, searchable description that covers:

1. Key facts, numbers, and data points from text and tables
2. Main topics and concepts discussed
3. Questions this content could answer
4. Visual content analysis (charts, diagrams, patterns in images)
5. Alternative search terms users might use

Make it detailed and searchable - prioritize findability over brevity.

SEARCHABLE DESCRIPTION:"""
    
    model = genai.GenerativeModel("gemini-2.5-flash-lite")  # ✅ Free tier, fast, vision support
    
    # Build parts list
    parts = [prompt_text]
    
    # Add images by converting base64 to PIL
    for image_base64 in images:
        image_bytes = base64.b64decode(image_base64)
        image = PIL.Image.open(io.BytesIO(image_bytes))
        parts.append(image)  # Gemini accepts PIL images directly

    response = model.generate_content(parts)
    return response.text.strip()                

def summarize_chunks(chunks) : 
    """Process all the chunks with ai summaries"""
    print("*"*5 + "Processing chunks with Ai summaries" + "*"*5)
    langchain_docs = []
    for i,chunk in enumerate(chunks) : 
        print("*"*3 +f"Processing chunk {i+1} / {len(chunks)}"+ "*"*3 )

        #analyze chunk content
        content_data = seperate_content_types(chunk)

        # debug print
        print(f"    Types found : {content_data['types']}")
        print(f"    Tables : {len(content_data['tables'])}, images : {len(content_data['images'])}")

        # create Ai-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images'] : 
            print(f"    Creating Ai summary for mixed content ....")
            try:
                enhanced_content=create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print(f"    Ai summary created successfully")
                print(f"    Enhanced content preview : {enhanced_content[:200]} ...")
            except Exception as e :
                print(f"    Ai summary failed : {e}")
                enhanced_content = content_data['text']
        else : 
            print(f"    Using raw text (no tables/images)")
            enhanced_content= content_data['text']
        
        # create the langchain document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content" : json.dumps({
                    "raw_text" : content_data['text'], 
                    "tables_html":content_data['tables'],
                    "images_base64":content_data['images'],
                })
            }
        )
        langchain_docs.append(doc)

    print(f"Processed {len(langchain_docs)} chunks")
    return langchain_docs

In [15]:
chunks[4].metadata.orig_elements

In [16]:
processed_chunks = summarize_chunks(chunks)

*****Processing chunks with Ai summaries*****
***Processing chunk 1 / 33***
    Types found : ['text']
    Tables : 0, images : 0
    Using raw text (no tables/images)
***Processing chunk 2 / 33***
    Types found : ['text']
    Tables : 0, images : 0
    Using raw text (no tables/images)
***Processing chunk 3 / 33***
    Types found : ['text']
    Tables : 0, images : 0
    Using raw text (no tables/images)
***Processing chunk 4 / 33***
    Types found : ['text']
    Tables : 0, images : 0
    Using raw text (no tables/images)
***Processing chunk 5 / 33***
    Types found : ['text', 'image']
    Tables : 0, images : 1
    Creating Ai summary for mixed content ....
    Ai summary created successfully
    Enhanced content preview : Here's a searchable description for the provided content:

**Key Facts, Numbers, and Data Points:**

*   **Model Structure:** Encoder-decoder architecture.
*   **Encoder Function:** Maps input sequenc ...
***Processing chunk 6 / 33***
    Types found : ['text

In [17]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "3\\n\\n2023\\n\\n2\\n\\n0\\n\\n2\\n\\ng u A 2 ] L C . s c [ 7 v 2 6 7 3 0 . 6 0 7\\n\\n1\\n\\n:\\n\\nv\\n\\narXiv\\n\\ni\\n\\nX\\n\\nr\\n\\na\\n\\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\\n\\nAttention Is All You Need\\n\\nAshish Vaswani\\u2217\\n\\nGoogle Brain\\n\\navaswani@google.com\\n\\nNoam Shazeer\\u2217\\n\\nGoogle Brain noam@google.com\\n\\nNiki Parmar\\u2217\\n\\nGoogle Research nikip@google.com\\n\\nJakob Uszkoreit\\u2217\\n\\nGoogle Research usz@google.com\\n\\nLlion Jones\\u2217\\n\\nGoogle Research llion@google.com\\n\\nAidan N. Gomez\\u2217 \\u2020 University of Toronto aidan@cs.toronto.edu\\n\\n\\u0141ukasz Kaiser\\u2217 Google Brain lukaszkaiser@google.com", "tables_html": [], "images_base64": []}'}, page_content='3\n\n2023\n\n2\n\n0\n\n2\n\ng u A 2 ] L C . s c [ 7 v 2 6 7 3 0 . 6 0 

In [18]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 33 chunks to chunks_export.json


In [20]:
def create_vector_store(documents, persist_directory="db/chroma_db_advanced") :
    """
    Create a vector store from the document chunks and persist it to disk.
    """

    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_name="multi_model_rag_collection",   # ADD THIS
        collection_metadata={"hnsw:space": "cosine"}
    )
    
    print (f"finished creating the vector store and persisting it to disk at {persist_directory}")
    return vector_store
db = create_vector_store(processed_chunks)


finished creating the vector store and persisting it to disk at db/chroma_db_advanced
